# Tutorial 6: Train NicheTrans* on STARmap PLUS data

In [1]:
import os, time, datetime, warnings

import torch
import torch.nn as nn
from torch.optim import lr_scheduler

from model.nicheTrans_ct import *
from datasets.data_manager_STARmap_PLUS import AD_Mouse

from utils.utils import *
from utils.utils_training_STARmap_PLUS import train, test
from utils.utils_dataloader import *

warnings.filterwarnings("ignore")

### Initialize the args and fix seeds

In [2]:
%run ./args/args_STARmap_PLUS.py
args = args

set_seed(args.seed)
os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_devices

print("==========\nArgs:{}\n==========".format(args))

Args:Namespace(noise_rate=0.5, dropout_rate=0.25, n_top_genes=2000, workers=4, AD_adata_path='/mnt/datadisk0/Processed_DATA/2023_nn_AD_mouse/AD_model_adata_protein', Wild_type_adata_path='/mnt/datadisk0/Processed_DATA/2023_nn_AD_mouse/wild_type_adata_protein', max_epoch=20, stepsize=20, train_batch=128, test_batch=32, optimizer='adam', lr=0.0001, gamma=0.1, weight_decay=0.0005, seed=1, save_dir='./log', eval_step=1, gpu_devices='0')


### Initialize dataloaders and NicheTrans

In [3]:
# create the dataloaders
dataset = AD_Mouse(AD_adata_path=args.AD_adata_path, Wild_type_adata_path=args.Wild_type_adata_path, n_top_genes=args.n_top_genes)
trainloader, testloader, _ = ad_mouse_dataloader(args, dataset)

# create the model
source_dimension, target_dimension = dataset.rna_length, dataset.target_length
model = NicheTrans_ct(source_length=source_dimension, target_length=target_dimension, noise_rate=args.noise_rate, dropout_rate=args.dropout_rate)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

------Calculating spatial graph...


The graph contains 124464 edges, 10372 cells.
12.0000 neighbors per cell on average.


------Calculating spatial graph...


The graph contains 115608 edges, 9634 cells.
12.0000 neighbors per cell on average.


------Calculating spatial graph...


The graph contains 96408 edges, 8034 cells.
12.0000 neighbors per cell on average.


=> AD Mouse loaded
Dataset statistics:
  ------------------------------
  subset   | # num | 
  ------------------------------
  train    |  10372 spots, 894.0 positive tao, 291.0 positive plaque 
  test     |   9634 spots, 620.0 positive tao, 195.0 positive plaque 
  ------------------------------


### Initialize loss function (criterion) and optimizer

In [4]:
criterion = nn.BCELoss()

if args.optimizer == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
elif args.optimizer == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=args.lr)
else:
    print('unexpected optimizer')

if args.stepsize > 0:
    scheduler = lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)

### Model training and testing

In [5]:
start_time = time.time()

for epoch in range(args.max_epoch):
    last_epoch = epoch + 1 == args.max_epoch

    print("==> Epoch {}/{}".format(epoch+1, args.max_epoch))
    
    ################
    train(model, criterion, optimizer, trainloader, ct_information=True, device=device)
    if args.stepsize > 0: scheduler.step()
    ################

pearson = test(model, testloader, ct_information=True, device=device)
torch.save(model.state_dict(), 'NicheTrans_*_STARmap_PLUS.pth')

elapsed = round(time.time() - start_time)
elapsed = str(datetime.timedelta(seconds=elapsed))
print("Finished. Total elapsed time (h:m:s): {}".format(elapsed))

==> Epoch 1/20


Batch 81/81	 Loss 0.587023 (0.666947)
==> Epoch 2/20


Batch 81/81	 Loss 0.442975 (0.519392)
==> Epoch 3/20


Batch 81/81	 Loss 0.368096 (0.418019)
==> Epoch 4/20


Batch 81/81	 Loss 0.322986 (0.351071)
==> Epoch 5/20


...

Batch 81/81	 Loss 0.109939 (0.112373)
==> Epoch 18/20


Batch 81/81	 Loss 0.095657 (0.108584)
==> Epoch 19/20


Batch 81/81	 Loss 0.100171 (0.103492)
==> Epoch 20/20


Batch 81/81	 Loss 0.078180 (0.097138)


tau AUC: 0.8193868677397883, tau sensitivity 0.5161290322580645, tay specificity 0.9348790769913468
Aβ AUC: 0.9270104123372478, Aβ sensitivity 0.676923076923077, Aβ specificity 0.9758448988240279
Finished. Total elapsed time (h:m:s): 0:34:14
